In [1]:
import json
import re
import uuid
from collections import defaultdict

# ---------------------------
# STEP 1: CLEAN TEXT
# ---------------------------
def clean_text(text):
    # remove reference numbers like [1], [23]
    text = re.sub(r"\[\d+\]", "", text)

    # remove extra spaces/newlines
    text = re.sub(r"\n+", "\n", text).strip()

    return text


# ---------------------------
# STEP 2: SPLIT INTO PARAGRAPHS
# ---------------------------
def split_paragraphs(text):
    paragraphs = [p.strip() for p in text.split("\n") if len(p.strip()) > 40]
    return paragraphs


# ---------------------------
# STEP 3: ORGAN DETECTION
# ---------------------------
def detect_organ(text):
    text = text.lower()

    if "heart" in text:
        return "heart"
    elif "brain" in text:
        return "brain"
    elif "neuron" in text:
        return "neuron"
    elif "circulatory" in text or "blood" in text:
        return "circulatory_system"
    elif "lymph" in text:
        return "lymphatic_system"
    else:
        return "general_biology"


# ---------------------------
# STEP 4: TOPIC EXTRACTION
# ---------------------------
def extract_topic(text):
    # simple heuristic: first sentence keyword
    first_line = text.split(".")[0].lower()

    if "circulation" in first_line:
        return "circulation"
    elif "heart" in first_line:
        return "heart"
    elif "blood vessel" in first_line:
        return "blood vessels"
    elif "artery" in first_line:
        return "arteries"
    elif "vein" in first_line:
        return "veins"
    elif "capillary" in first_line:
        return "capillaries"
    elif "disease" in first_line:
        return "diseases"
    elif "development" in first_line:
        return "development"
    elif "function" in first_line:
        return "function"
    else:
        return "general"


# ---------------------------
# STEP 5: AUTO TAGGING
# ---------------------------
def generate_tags(text):
    text = text.lower()
    tags = set()

    if "is" in text or "are" in text:
        tags.add("definition")

    if "function" in text or "responsible" in text:
        tags.add("function")

    if "consists" in text or "includes" in text:
        tags.add("structure")

    if "types" in text or "classified" in text:
        tags.add("classification")

    if "flow" in text or "circulation" in text:
        tags.add("process")

    if "disease" in text or "disorder" in text:
        tags.add("disease")

    if "develop" in text or "embryo" in text:
        tags.add("development")

    if not tags:
        tags.add("concept")

    return list(tags)


# ---------------------------
# STEP 6: BUILD JSON
# ---------------------------
def convert_txt_to_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    cleaned = clean_text(raw_text)
    paragraphs = split_paragraphs(cleaned)

    output = []

    for para in paragraphs:
        entry = {
            "id": str(uuid.uuid4()),
            "organ": detect_organ(para),
            "topic": extract_topic(para),
            "text": para,
            "tags": generate_tags(para)
        }
        output.append(entry)

    return output


# ---------------------------
# STEP 7: SAVE FILE
# ---------------------------
def save_json(data, output_file="output.json"):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# ---------------------------
# RUN PIPELINE
# ---------------------------
if __name__ == "__main__":
    data = convert_txt_to_json("data/heart.txt")
    save_json(data)

    print(f"Generated {len(data)} structured entries.")

Generated 110 structured entries.


In [1]:
# =========================
# IMPORTS
# =========================
import json
import re
import uuid
import requests
from typing import List

# =========================
# CONFIG
# =========================
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "mistral"
BATCH_SIZE = 2   # keep small for better JSON accuracy


# =========================
# STEP 1: CLEAN TEXT
# =========================
def clean_text(text: str) -> str:
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()


# =========================
# STEP 2: SPLIT PARAGRAPHS
# =========================
def split_paragraphs(text: str) -> List[str]:
    return [p.strip() for p in text.split("\n") if len(p.strip()) > 50]


# =========================
# STEP 3: BATCHING
# =========================
def batch_paragraphs(paragraphs: List[str], batch_size: int):
    for i in range(0, len(paragraphs), batch_size):
        yield paragraphs[i:i + batch_size]


# =========================
# STEP 4: CLEAN JSON
# =========================
def clean_json(text: str):
    text = re.sub(r"```json|```", "", text).strip()
    try:
        return json.loads(text)
    except:
        return None


# =========================
# STEP 5: CALL OLLAMA
# =========================
def call_ollama(prompt: str):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]


# =========================
# STEP 6: EXTRACTION
# =========================
def extract_batch(batch: List[str]):
    joined_text = "\n\n".join(batch)

    prompt = f"""
You are a strict JSON generator.

Convert EACH paragraph into structured JSON.

Rules:
- Output ONLY JSON
- No explanation
- No markdown
- One object per paragraph

Format:
[
  {{
    "organ": "...",
    "topic": "...",
    "tags": ["definition", "function"]
  }}
]

Paragraphs:
{joined_text}
"""

    output = call_ollama(prompt)

    # Try parsing
    parsed = clean_json(output)

    # Retry once if failed
    if parsed is None:
        print("⚠️ Retrying with stricter prompt...")

        retry_prompt = f"""
STRICT JSON ONLY. NO TEXT.

{prompt}
"""
        output = call_ollama(retry_prompt)
        parsed = clean_json(output)

    if parsed is None:
        print("❌ Failed batch. Skipping.")
        return []

    return parsed


# =========================
# STEP 7: POST PROCESS
# =========================
def post_process(entries: List[dict], original_batch: List[str]):
    processed = []

    for i, entry in enumerate(entries):
        try:
            entry["id"] = str(uuid.uuid4())
            entry["text"] = original_batch[i]
            processed.append(entry)
        except:
            continue

    return processed


# =========================
# STEP 8: MAIN PIPELINE
# =========================
def txt_to_structured_json(input_file: str, output_file: str):
    with open(input_file, "r", encoding="utf-8") as f:
        raw_text = f.read()

    print("🔹 Cleaning text...")
    cleaned = clean_text(raw_text)

    print("🔹 Splitting paragraphs...")
    paragraphs = split_paragraphs(cleaned)

    print(f"🔹 Total paragraphs: {len(paragraphs)}")

    all_results = []

    for idx, batch in enumerate(batch_paragraphs(paragraphs, BATCH_SIZE)):
        print(f"⚡ Processing batch {idx + 1}...")

        extracted = extract_batch(batch)
        processed = post_process(extracted, batch)

        all_results.extend(processed)

    print("🔹 Saving JSON...")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print(f"✅ Done! Generated {len(all_results)} entries.")


# =========================
# RUN
# =========================
# txt_to_structured_json("data/heart.txt", "circulatory.json")
txt_to_structured_json("data/neural.txt", "nervous.json")

🔹 Cleaning text...
🔹 Splitting paragraphs...
🔹 Total paragraphs: 65
⚡ Processing batch 1...
⚡ Processing batch 2...
⚡ Processing batch 3...
⚡ Processing batch 4...
⚡ Processing batch 5...
⚡ Processing batch 6...
⚡ Processing batch 7...
⚡ Processing batch 8...
⚡ Processing batch 9...
⚡ Processing batch 10...
⚡ Processing batch 11...
⚡ Processing batch 12...
⚡ Processing batch 13...
⚡ Processing batch 14...
⚡ Processing batch 15...
⚡ Processing batch 16...
⚡ Processing batch 17...
⚡ Processing batch 18...
⚡ Processing batch 19...
⚡ Processing batch 20...
⚡ Processing batch 21...
⚡ Processing batch 22...
⚡ Processing batch 23...
⚡ Processing batch 24...
⚡ Processing batch 25...
⚡ Processing batch 26...
⚡ Processing batch 27...
⚡ Processing batch 28...
⚡ Processing batch 29...
⚡ Processing batch 30...
⚡ Processing batch 31...
⚡ Processing batch 32...
⚡ Processing batch 33...
🔹 Saving JSON...
✅ Done! Generated 65 entries.
